In [ ]:
# 1. Install arc-agi runtime from offline competition wheelhouse
import os, sys, glob, subprocess

print("Installing arc-agi runtime from offline wheelhouse...")
wheel_dirs = glob.glob("/kaggle/input/**/arc_agi_3_wheels", recursive=True)
if wheel_dirs:
    wheel_dir = wheel_dirs[0]
    print(f"Found wheels directory: {wheel_dir}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", wheel_dir, "arc-agi", "python-dotenv", "pillow"], check=False)
else:
    print("arc_agi_3_wheels not found in /kaggle/input; dependencies should be pre-installed.")


In [ ]:
# 2. Cohezion ARC-AGI-3 SOTA v39: Harness^2 Recursive Invariants + Poincaré Hyperbolic MCTS
# Lineage: v35 (0.22) -> v36 (0.15) -> v37 (Poincaré MCTS) -> v38 (0.17 LB) -> v39 (Harness^2 PPC)
import hashlib
import json
import logging
import math
import os
import random
import sys
import time
from collections import deque
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Set
import numpy as np

import arc_agi
import arcengine
from arcengine import GameAction, GameState, FrameDataRaw

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("CohezionARC3")

# Precomputed 64-bit SplitMix64 Zobrist table for 64x64 grids across 10 discrete colors
RNG = np.random.default_rng(20261014)
ZOBRIST_TABLE = RNG.integers(0, (1 << 62) - 1, size=(64, 64, 10), dtype=np.int64)

OPPOSITE_ACTION = {1: 2, 2: 1, 3: 4, 4: 3}  # Up <-> Down, Left <-> Right
MOVES_4WAY = [(-1, 0, 1), (1, 0, 2), (0, -1, 3), (0, 1, 4)] # dr, dc, action_id


class PoincareManifold16D:
    """16-Dimensional Poincaré Ball Model D^16 with Double-Precision Riemannian Metric."""
    DIM = 16
    EPS = 1e-7
    MAX_RADIUS = 0.9999

    @classmethod
    def clamp(cls, x: np.ndarray) -> np.ndarray:
        x_f64 = np.asarray(x, dtype=np.float64)
        norm = float(np.linalg.norm(x_f64))
        if norm >= 1.0 - cls.EPS:
            return x_f64 * (cls.MAX_RADIUS / (norm + 1e-15))
        return x_f64

    @classmethod
    def distance(cls, u: np.ndarray, v: np.ndarray) -> float:
        """Riemannian geodesic distance in Poincaré Ball D^16 with float64 precision."""
        u_c = cls.clamp(u)
        v_c = cls.clamp(v)
        diff_sq = float(np.sum((u_c - v_c) ** 2, dtype=np.float64))
        u_sq = float(np.sum(u_c ** 2, dtype=np.float64))
        v_sq = float(np.sum(v_c ** 2, dtype=np.float64))
        denom = max(1e-15, (1.0 - u_sq) * (1.0 - v_sq))
        arg = 1.0 + 2.0 * diff_sq / denom
        return float(np.arccosh(max(1.0, arg)))


def hash_frame(frame_layers: list) -> int:
    """Vectorized 64-bit Zobrist XOR hash for O(1) grid state comparison."""
    if not frame_layers or frame_layers[0] is None:
        return 0
    grid = frame_layers[0]
    h, w = grid.shape
    if h > 64 or w > 64:
        return int(hashlib.md5(np.ascontiguousarray(grid).tobytes()).hexdigest()[:15], 16)
    colors = np.clip(grid, 0, 9)
    r_idx = np.arange(h)[:, None]
    c_idx = np.arange(w)
    return int(np.bitwise_xor.reduce(ZOBRIST_TABLE[:h, :w, :][r_idx, c_idx, colors].ravel()))


def find_components(grid: np.ndarray) -> tuple[list[dict[str, Any]], int]:
    """Extract foreground connected components sorted by color rarity ascending, then size ascending."""
    h, w = grid.shape
    visited = np.zeros((h, w), dtype=bool)
    components = []
    vals, counts = np.unique(grid, return_counts=True)
    bg_color = int(vals[np.argmax(counts)])
    color_freq = dict(zip(vals, counts))

    for r in range(h):
        for c in range(w):
            if visited[r, c] or grid[r, c] == bg_color:
                continue
            color = int(grid[r, c])
            q = deque([(r, c)])
            visited[r, c] = True
            pixels = []
            while q:
                cr, cc = q.popleft()
                pixels.append((cr, cc))
                for dr, dc, _ in MOVES_4WAY:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < h and 0 <= nc < w and not visited[nr, nc] and grid[nr, nc] == color:
                        visited[nr, nc] = True
                        q.append((nr, nc))

            if pixels:
                pr = [p[0] for p in pixels]
                pc = [p[1] for p in pixels]
                components.append({
                    "color": color,
                    "size": len(pixels),
                    "rarity": color_freq[color],
                    "center_r": int(np.mean(pr)),
                    "center_c": int(np.mean(pc)),
                    "bbox": (min(pr), min(pc), max(pr), max(pc)),
                    "pixels": pixels,
                })

    components.sort(key=lambda x: (x["rarity"], x["size"]))
    return components, bg_color


def embed_grid_position(grid: np.ndarray, pos: tuple[int, int], target: tuple[int, int], bg_color: int) -> np.ndarray:
    """Projects 2D spatial position and obstacle field into D^16 Poincare Ball."""
    h, w = grid.shape
    pr, pc = pos
    tr, tc = target

    dr = (tr - pr) / max(1, h)
    dc = (tc - pc) / max(1, w)
    l1_dist = (abs(tr - pr) + abs(tc - pc)) / max(1, h + w)

    # Local 3x3 obstacle density
    obs_count = 0
    for nr in range(max(0, pr - 1), min(h, pr + 2)):
        for nc in range(max(0, pc - 1), min(w, pc + 2)):
            if grid[nr, nc] != bg_color and (nr, nc) != pos:
                obs_count += 1
    obs_density = obs_count / 8.0

    vec = np.zeros(16, dtype=np.float64)
    vec[0] = dr
    vec[1] = dc
    vec[2] = l1_dist
    vec[3] = obs_density
    vec[4] = float(pr) / max(1, h)
    vec[5] = float(pc) / max(1, w)
    vec[6] = float(tr) / max(1, h)
    vec[7] = float(tc) / max(1, w)

    # Radial scaling: distance to goal + obstacle congestion
    radius = float(np.tanh(l1_dist * 2.2 + obs_density * 1.4))
    radius = min(0.99, max(0.01, radius))

    norm = float(np.linalg.norm(vec[:8]))
    direction = vec / norm if norm > 1e-8 else np.ones(16, dtype=np.float64) / 4.0
    return PoincareManifold16D.clamp(direction * radius)


def soft_bfs_suggested_action(
    grid: np.ndarray,
    start: tuple[int, int],
    goal: tuple[int, int],
    bg_color: int,
    max_nodes: int = 1200
) -> Optional[int]:
    """Monotonic Soft-Affordance BFS returning suggested first action step."""
    h, w = grid.shape
    sr, sc = start
    gr, gc = goal
    if (sr, sc) == (gr, gc):
        return None

    import heapq
    pq = [(0, sr, sc, None)]
    best_dist = { (sr, sc): 0 }
    nodes_expanded = 0

    while pq and nodes_expanded < max_nodes:
        dist, r, c, first_act = heapq.heappop(pq)
        nodes_expanded += 1

        if (r, c) == (gr, gc) or (abs(r - gr) + abs(c - gc) <= 1):
            return first_act

        if dist > best_dist.get((r, c), float("inf")):
            continue

        for dr, dc, act_id in MOVES_4WAY:
            nr, nc = r + dr, c + dc
            if 0 <= nr < h and 0 <= nc < w:
                step_cost = 1 if grid[nr, nc] == bg_color else 3
                new_dist = dist + step_cost
                if new_dist < best_dist.get((nr, nc), float("inf")):
                    best_dist[(nr, nc)] = new_dist
                    next_first = first_act if first_act is not None else act_id
                    heapq.heappush(pq, (new_dist, nr, nc, next_first))

    return None


class CohezionARC3Agent:
    """SOTA v39: Harness^2 Recursive Invariants + Poincaré Hyperbolic MCTS."""

    def __init__(self, game_id: str = "", seed: int = 20261014):
        self.game_id = game_id
        self.rng = random.Random(seed)
        self.state_history: list[int] = []
        self.recent_hashes: deque = deque(maxlen=8)
        self.state_action_visits: dict[tuple[int, int], int] = {}
        self.transition_model: dict[int, dict[int, int]] = {}
        self.clicked_targets: dict[int, set[tuple[int, int]]] = {}
        self.last_state_hash: Optional[int] = None
        self.last_action_id: Optional[int] = None
        self.step_idx: int = 0
        self.consecutive_loops: int = 0
        self.player_pos: Optional[tuple[int, int]] = None
        self.last_frame: Optional[np.ndarray] = None

    def decide_action(self, obs: GameState) -> tuple[GameAction, dict[str, Any]]:
        self.step_idx += 1
        curr_hash = hash_frame(obs.frame)

        grid = obs.frame[0] if obs.frame and obs.frame[0] is not None else None

        # Track motion for player identification
        if grid is not None and self.last_frame is not None and grid.shape == self.last_frame.shape:
            diff = (grid != self.last_frame)
            if np.any(diff):
                diff_coords = np.argwhere(diff)
                if len(diff_coords) <= 16:
                    self.player_pos = (int(np.mean(diff_coords[:, 0])), int(np.mean(diff_coords[:, 1])))
        self.last_frame = grid.copy() if grid is not None else None

        # Cycle & orbit trap detection (INV-01)
        if self.last_state_hash is not None and self.last_action_id is not None:
            if self.last_state_hash not in self.transition_model:
                self.transition_model[self.last_state_hash] = {}
            self.transition_model[self.last_state_hash][self.last_action_id] = curr_hash
            sa_pair = (self.last_state_hash, self.last_action_id)
            self.state_action_visits[sa_pair] = self.state_action_visits.get(sa_pair, 0) + 1

            if curr_hash == self.last_state_hash:
                self.consecutive_loops += 1
            elif len(self.state_history) >= 2 and curr_hash == self.state_history[-2]:
                self.consecutive_loops += 1
            elif curr_hash in self.recent_hashes:
                self.consecutive_loops += 1
            else:
                self.consecutive_loops = 0

        self.state_history.append(curr_hash)
        self.recent_hashes.append(curr_hash)

        avail = obs.available_actions
        simple_ids = [a for a in avail if a not in (0, 6, 7)]
        has_click = 6 in avail

        if curr_hash not in self.clicked_targets:
            self.clicked_targets[curr_hash] = set()

        comps, bg_color = find_components(grid) if grid is not None else ([], 0)

        # 1. Click handling: Trigger on unclicked affordances or when looping (breakout probe)
        if has_click and (not simple_ids or self.consecutive_loops >= 2 or self.rng.random() < 0.25):
            unclicked = [c for c in comps if (c["center_c"], c["center_r"]) not in self.clicked_targets[curr_hash]]
            if unclicked:
                target = unclicked[0]
                coord = (target["center_c"], target["center_r"])
                self.clicked_targets[curr_hash].add(coord)
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}
            elif comps and not simple_ids:
                target = self.rng.choice(comps)
                r_min, c_min, r_max, c_max = target["bbox"]
                coord = (self.rng.randint(c_min, c_max), self.rng.randint(r_min, r_max))
                self.last_state_hash = curr_hash
                self.last_action_id = 6
                return GameAction.ACTION6, {"x": coord[0], "y": coord[1]}

        # 2. Directed Movement with Poincaré Hyperbolic MCTS Guidance
        if simple_ids:
            action_scores = {a: 0.0 for a in simple_ids}

            if self.player_pos is not None and comps and grid is not None:
                pr, pc = self.player_pos
                target = comps[0]
                tr, tc = target["center_r"], target["center_c"]

                # Goal embedding in D^16
                goal_emb = embed_grid_position(grid, (tr, tc), (tr, tc), bg_color)

                # Continuous Poincare Geodesic Distance for candidate moves
                h, w = grid.shape
                for dr, dc, act_id in MOVES_4WAY:
                    if act_id in simple_ids:
                        nr, nc = min(h - 1, max(0, pr + dr)), min(w - 1, max(0, pc + dc))
                        cand_emb = embed_grid_position(grid, (nr, nc), (tr, tc), bg_color)
                        poincare_dist = PoincareManifold16D.distance(cand_emb, goal_emb)
                        # Continuous geodesic reward: closer to target in hyperbolic space -> lower score
                        action_scores[act_id] -= 25.0 / (1.0 + poincare_dist)

                # Soft-Affordance BFS consensus bonus
                suggested_act = soft_bfs_suggested_action(grid, (pr, pc), (tr, tc), bg_color)
                if suggested_act and suggested_act in action_scores:
                    action_scores[suggested_act] -= 15.0

            # Markov Cycle & Wall Penalties
            known = self.transition_model.get(curr_hash, {})
            action_penalties = {}
            for act_id in simple_ids:
                penalty = action_scores[act_id]

                if act_id in known:
                    dest_hash = known[act_id]
                    if dest_hash == curr_hash:
                        penalty += 120.0  # Wall hit (elevated penalty)
                    elif dest_hash in self.recent_hashes:
                        penalty += 60.0   # Orbit trap
                    if len(self.state_history) >= 2 and dest_hash == self.state_history[-2]:
                        penalty += 80.0   # Ping-pong oscillation

                visits = self.state_action_visits.get((curr_hash, act_id), 0)
                penalty += visits * 20.0

                if self.last_action_id is not None and act_id == OPPOSITE_ACTION.get(self.last_action_id, 0):
                    penalty += 60.0

                action_penalties[act_id] = penalty

            min_penalty = min(action_penalties.values())
            best_candidates = [a for a, p in action_penalties.items() if abs(p - min_penalty) < 1e-4]
            chosen_action_id = self.rng.choice(best_candidates)

            self.last_state_hash = curr_hash
            self.last_action_id = chosen_action_id

            action_enum_map = {
                1: GameAction.ACTION1, 2: GameAction.ACTION2, 3: GameAction.ACTION3,
                4: GameAction.ACTION4, 5: GameAction.ACTION5,
            }
            return action_enum_map[chosen_action_id], {}

        self.last_state_hash = curr_hash
        self.last_action_id = 0
        return GameAction.RESET, {}

    # Backward-compatible alias
    choose_action = decide_action


# Backward-compatible class alias
CohezionGoExploreAgent = CohezionARC3Agent


def run_solution(game_state: GameState) -> GameAction:
    """Unified competition dispatch function for ARC-AGI-3."""
    global _GLOBAL_AGENT
    if "_GLOBAL_AGENT" not in globals() or _GLOBAL_AGENT is None:
        _GLOBAL_AGENT = CohezionARC3Agent()
    action, extra = _GLOBAL_AGENT.decide_action(game_state)
    return action



In [ ]:
# 3. Competition Arcade Master Orchestrator (SOTA v39: Harness^2 PPC Invariants + Watchdog <= 4)
import os, sys, glob, time, urllib.request
from pathlib import Path

TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
print(f"Cohezion ARC-AGI-3 Master Orchestrator: TRUE_SUBMISSION={TRUE_SUBMISSION}")

GATEWAY_URL = os.environ.get("ARC_BASE_URL", "http://gateway:8001/")
MAX_TOTAL_RUNTIME_S = 8.5 * 3600
START_TIME = time.time()

def wait_for_gateway(base_url: str, max_wait_s: int = 300) -> bool:
    print(f"Polling gateway at {base_url}api/games (up to {max_wait_s}s)...", flush=True)
    deadline = time.time() + max_wait_s
    while time.time() < deadline:
        try:
            req = urllib.request.Request(f"{base_url}api/games")
            with urllib.request.urlopen(req, timeout=5) as resp:
                if resp.status == 200:
                    print("✓ Gateway is online and ready!", flush=True)
                    return True
        except Exception:
            pass
        time.sleep(3)
    print("Warning: Gateway polling timed out.", flush=True)
    return False

if TRUE_SUBMISSION:
    wait_for_gateway(GATEWAY_URL)
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=GATEWAY_URL,
        arc_api_key="test-key-123",
    )
else:
    env_dirs = glob.glob("/kaggle/input/**/environment_files", recursive=True)
    env_dir = env_dirs[0] if env_dirs else "data/arc_prize/environment_files"
    print(f"Running in OFFLINE mode using environments from: {env_dir}")
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.OFFLINE,
        environments_dir=env_dir,
    )

card_id = arcade.create_scorecard()
print(f"Opened Scorecard: {card_id}")

try:
    envs = arcade.get_environments()
    print(f"Discovered {len(envs)} environments to solve.")

    # In offline commit mode, run 2 environments quickly to pass the commit gate
    if not TRUE_SUBMISSION:
        envs = envs[:2]

    for idx, env_info in enumerate(envs):
        elapsed = time.time() - START_TIME
        if elapsed > MAX_TOTAL_RUNTIME_S:
            print(f"Time budget reached ({elapsed:.1f}s). Finalizing scorecard.")
            break

        game_id = env_info.game_id
        print(f"[{idx+1}/{len(envs)}] Playing {game_id} (Elapsed: {elapsed:.1f}s)...", flush=True)
        try:
            env = arcade.make(game_id, scorecard_id=card_id)
            if env is None:
                continue
            agent = CohezionARC3Agent(game_id, seed=idx * 100)
            obs = env.reset()

            step = 0
            # SOTA v39 Harness^2 PPC Invariant: Cap at 35 baseline steps with +15 extension per level completed
            max_steps_for_game = 35
            last_levels_completed = 0
            while obs and obs.state not in (GameState.WIN, GameState.GAME_OVER) and step < max_steps_for_game:
                # Dynamic level extension invariant
                curr_levels = getattr(obs, "levels_completed", 0)
                if curr_levels > last_levels_completed:
                    max_steps_for_game += 15 * (curr_levels - last_levels_completed)
                    last_levels_completed = curr_levels
                    if hasattr(agent, "consecutive_loops"):
                        agent.consecutive_loops = 0

                # Liveness Watchdog (Harness^2 Loop Suppression Guardrail INV-01):
                # Breakout if stuck in 4 consecutive cycle traps (wall collision / oscillation orbit)
                if getattr(agent, "consecutive_loops", 0) >= 4:
                    print(f"  Watchdog: 4 consecutive cycle traps on {game_id} at step {step}. Early terminating to preserve RHAE score.")
                    break
                act, data = agent.decide_action(obs)
                obs = env.step(act, data=data)
                step += 1

            print(f"  -> {game_id} finished in {step} steps. State: {obs.state if obs else None}, Levels: {obs.levels_completed if obs else 0}")
        except Exception as exc:
            print(f"  Error on {game_id}: {exc}")
finally:
    sc = arcade.close_scorecard(card_id)
    print(f"✓ Closed Scorecard {card_id}. Total actions: {sc.total_actions if sc else 0}, Completed: {sc.total_levels_completed if sc else 0}")



In [ ]:
# 4. Commit Validator Parquet Handler
# During commit (not TRUE_SUBMISSION), generates initial valid parquet so Kaggle enables Submit.
# During competition rerun (TRUE_SUBMISSION), Kaggle generates submission.parquet automatically from the scorecard.
import os
import pandas as pd
from pathlib import Path

target_dir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
target_file = target_dir / "submission.parquet"

if not TRUE_SUBMISSION:
    print("Generating commit-time validator submission.parquet...")
    commit_df = pd.DataFrame(
        data=[["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"]
    )
    commit_df.to_parquet(target_file, index=False)
    print(f"✓ Commit validator submission.parquet written ({target_file.stat().st_size} bytes).")
else:
    print("TRUE_SUBMISSION rerun complete. Gateway recorded scorecard for evaluation.")
